# CSP baseline versus TSF

A minimal train/test comparison of three probabilistic forecasts:

- **CSP**: training-free probabilistic baseline;
- **TSF without features**: lags and calendar only;
- **TSF with features**: lags, calendar, and a promotion known in advance.

The PDF calls the baseline **Conformal Seasonal Pool (CSP)**.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from mlforecast import MLForecast
from sklearn.ensemble import HistGradientBoostingRegressor
from statsforecast.models import ConformalSeasonalPool

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
sys.path.insert(0, str(project_root))
from tinyshift.forecasting import (
    GammaFamily,
    TwoStageForecasterWrapper,
    crps_distribution,
    crps_ensemble,
    mwis,
    ncrps,
)

/home/heylucasleao/tinyshift/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Train and test data

Demand has weekly seasonality and a real promotion effect. The last 28 days form the test set.

In [2]:
rng = np.random.default_rng(42)
n = 420
df = pd.DataFrame({"unique_id": "item_1", "ds": pd.date_range("2025-01-01", periods=n)})
dow = df["ds"].dt.dayofweek
df["promo"] = rng.binomial(1, 0.12, n)
df["y"] = 50 + 8 * np.sin(2 * np.pi * dow / 7) + 15 * df["promo"] + rng.normal(0, 4, n)

horizon = 28
train = df.iloc[:-horizon].copy()
test = df.iloc[-horizon:].copy()
train.shape, test.shape

((392, 4), (28, 4))

## Fit the three forecasts

TSF uses a Gamma predictive distribution because the target is positive and continuous. Its dispersion is calibrated from temporal out-of-fold errors inside the training set; the test set is used only for final evaluation.

In [3]:
probabilities = np.linspace(0.005, 0.995, 199)
forecasts = {}

csp = ConformalSeasonalPool(season_length=7, n_samples=5000, variant="adaptive")
csp.fit(train["y"].to_numpy())
csp_samples = csp.simulate(h=horizon, n_paths=5000, seed=42)
forecasts["CSP"] = {
    "mean": csp.predict(horizon)["mean"],
    "samples": csp_samples.T,
    "quantiles": np.quantile(csp_samples, probabilities, axis=0).T,
}

for name, features in {"TSF without features": [], "TSF with features": ["promo"]}.items():
    mlforecast = MLForecast(
        models={name: HistGradientBoostingRegressor(max_iter=150, max_depth=3, random_state=42)},
        freq="D", lags=[14, 28], date_features=["dayofweek"],
    )
    model = TwoStageForecasterWrapper(mlforecast, distribution=GammaFamily()).fit(
        train[["unique_id", "ds", "y", *features]],
        static_features=[], horizon=7, n_windows=7, step_size=28,
    )
    future_x = test[["unique_id", "ds", *features]] if features else None
    forecast = model.predict_distribution(h=horizon, X_df=future_x)
    forecasts[name] = {
        "mean": forecast.to_frame()["lambda_t"].to_numpy(),
        "distribution": forecast.distribution,
        "quantiles": forecast.distribution.ppf(probabilities),
    }

## Test metrics

RMSE evaluates the point mean. CRPS evaluates the complete predictive distribution, and nCRPS divides it by the target standard deviation in training. Coverage should be close to 90%; MWIS rewards narrow intervals but penalizes misses. Lower is better for every metric except coverage.

In [4]:
y = test["y"].to_numpy()
train_scale = train["y"].std(ddof=1)
rows = []
for name, forecast in forecasts.items():
    mean, quantiles = forecast["mean"], forecast["quantiles"]
    row_crps = (
        crps_ensemble(y, forecast["samples"])
        if name == "CSP"
        else crps_distribution(y, forecast["distribution"])
    )
    lower = np.array([np.interp(0.05, probabilities, row) for row in quantiles])
    upper = np.array([np.interp(0.95, probabilities, row) for row in quantiles])
    rows.append({
        "model": name,
        "rmse": np.sqrt(np.mean((y - mean) ** 2)),
        "crps": row_crps.mean(),
        "ncrps": ncrps(row_crps.mean(), train_scale).item(),
        "coverage_90": np.mean((y >= lower) & (y <= upper)),
        "mwis_90": mwis(y, lower, upper, alpha=0.10),
    })

summary = pd.DataFrame(rows).set_index("model").round(3)
summary

,rmse,crps,ncrps,coverage_90,mwis_90
model,,,,,
CSP,8.199,3.459,0.419,1.000,25.734
TSF without features,6.986,3.921,0.475,0.857,30.129
TSF with features,3.690,2.217,0.269,0.964,17.443
